# Road segments processing

import packages

In [ ]:
import geopandas as gpd
import pandas as pd
import fiona
import networkx as nx
from shapely.geometry import MultiLineString, LineString
from shapely.ops import unary_union, linemerge
import os
import time

# Enable KML driver
fiona.drvsupport.supported_drivers['KML'] = 'rw'


define global variables 

In [ ]:
os.chdir('..')
os.getcwd()

In [ ]:
shp_path="data/raw/road_routes/geometria/Geometria_tramos.shp"
traffic_path="data/processed/road_routes_traffic.parquet"
kmz_path="data/raw/roads/query.kmz"
output_path="data/processed/integrated_road_network.parquet"
small_segment_length_m=2000
bridge_gap_threshold_m=1000
parallel_threshold_m=100

loading segments data

In [ ]:
gdf_segments = gpd.read_file(shp_path)

In [ ]:
print("Loading data...")
# Load Shapefile
gdf_segments = gpd.read_file(shp_path)
gdf_segments['id_tramo'] = gdf_segments['id_tramo'].astype(str)

print(f"Loaded {len(gdf_segments)} segments.")
gdf_segments


loading segments info 

In [ ]:
# Load CSV
df_info = pd.read_parquet(traffic_path)
df_info['tramo'] = df_info['tramo'].astype(str)
print(f"Loading CSV... {len(df_info)} segments info.")
df_info

In [ ]:
traffic_cols = [c for c in df_info.columns if c.startswith('total_')]
if not traffic_cols:
    print("Warning: No 'total_YYYYMMDD' columns found.")

gdf_segments['id_tramo'] = gdf_segments['id_tramo'].astype(str)

loading roads backbones KMZ  

In [ ]:
# Load KMZ as backbone reference
gdf_backbone = gpd.read_file(kmz_path, driver='KML')
gdf_backbone = gdf_backbone.to_crs(gdf_segments.crs)
gdf_backbone['backbone_id'] = gdf_backbone.index
print(f"KMZ road backbones loaded {len(gdf_backbone)}")
gdf_backbone = gdf_backbone[['backbone_id','geometry']]
gdf_backbone

In [ ]:
# Filter out small backbone reference roads
gdf_backbone['backbone_length_m'] = gdf_backbone.geometry.length
gdf_backbone = gdf_backbone[gdf_backbone['backbone_length_m'] >= small_segment_length_m].copy()
print(f"Filtered KMZ: {len(gdf_backbone)} backbones remain (min length {small_segment_length_m}m).")

In [ ]:
print("Merging shapefile and traffic data")
gdf_merged = gdf_segments.merge(df_info, left_on='id_tramo', right_on='tramo')

if traffic_cols:
    gdf_merged['total'] = gdf_merged[traffic_cols].mean(axis=1)
else:
    gdf_merged['total'] = 0
    
gdf_merged['length_m'] = gdf_merged.geometry.length
gdf_merged

In [ ]:
gdf_centroids = gdf_merged.copy()
gdf_centroids['geometry'] = gdf_centroids.geometry.centroid

In [ ]:
gdf_assigned_centroids = gpd.sjoin_nearest(
        gdf_centroids, 
        gdf_backbone[['backbone_id', 'geometry']], 
        max_distance=500, # 2km max distance to road backbone
        distance_col="dist_to_backbone"
    )
  

In [ ]:
gdf_merged = gdf_merged.merge(
        gdf_assigned_centroids[['id_tramo', 'backbone_id', 'dist_to_backbone']], 
        on='id_tramo', 
        how='left'
    )
gdf_merged

In [ ]:
gdf_merged = gdf_merged.dropna(subset=['backbone_id'])
print(f"Beginning simplification for {len(gdf_merged)} matched segments...")

In [ ]:
processed_groups = []
backbone_groups = list(gdf_merged.groupby('backbone_id'))
total_groups = len(backbone_groups)

In [ ]:
start_time = time.time()
for i, (bib_id, group) in enumerate(backbone_groups):
    if i % 200 == 0:
        print(f"Processing backbone {i}/{total_groups}... (Elapsed: {time.time()-start_time:.1f}s)")

In [ ]:
gdf_backbone['backbone_id'].unique()

In [ ]:
# Select the Backbone and its assigned segments
bib_id = 1587
selected_backbone = gdf_backbone[gdf_backbone['backbone_id'] == bib_id]
selected_backbone = selected_backbone.to_crs(epsg=4326)
selected_segments = gdf_merged[gdf_merged['backbone_id'] == bib_id]
selected_segments = selected_segments.to_crs(epsg=4326)


In [ ]:
import folium 

# Initialize map at the center of the backbone
center = selected_backbone.geometry.union_all().centroid
m = folium.Map(location=[center.y, center.x], zoom_start=14, tiles='CartoDB positron')


folium.GeoJson(
    selected_segments,
    name="Assigned Segments",
    style_function=lambda x: {'color': 'red', 'weight': 3, 'opacity': 0.7}
).add_to(m)

m

In [ ]:
folium.GeoJson(
    selected_backbone,
    name="KMZ Backbone Reference",
    style_function=lambda x: {'color': 'blue', 'weight': 5, 'opacity': 0.9}
).add_to(m)

m


In [ ]:
group = group.copy().reset_index(drop=True)

In [ ]:
G = nx.Graph()
G.add_nodes_from(group.index)

# Spatial Index query
sindex = group.sindex

# We only need to check segments against their neighbors
for idx in group.index:
    geom = group.at[idx, 'geometry']
    length = group.at[idx, 'length_m']
    
    # Max possible search radius
    search_rad = max(bridge_gap_threshold_m, parallel_threshold_m)
    possible_indices = sindex.query(geom.buffer(search_rad))
    
    for other_idx in possible_indices:
        if idx >= other_idx:
            continue
        
        other_geom = group.at[other_idx, 'geometry']
        other_length = group.at[other_idx, 'length_m']
        dist = geom.distance(other_geom)
        
        if dist < parallel_threshold_m:
            G.add_edge(idx, other_idx)
        elif (length < small_segment_length_m or other_length < small_segment_length_m) and dist < bridge_gap_threshold_m:
            G.add_edge(idx, other_idx)

In [ ]:
for component in nx.connected_components(G):
    print(component)

In [ ]:
subset = group.iloc[list(component)]

In [ ]:
merged_geom = unary_union(subset.geometry)

In [ ]:
if not merged_geom.is_empty:
    print('not empty union')
    if isinstance(merged_geom, (MultiLineString, list)):
        print('merged_geom is: MultiLineString, list')
        try:
            merged_geom = linemerge(merged_geom)
            print('linemerge merged_geom')
        except Exception:
            pass # Keep as MultiLineString if merge fail

In [ ]:
traffic_sums = subset[traffic_cols].sum().to_dict()
traffic_sums

In [ ]:
processed_entry = {
    'backbone_id': bib_id,
    'geometry': merged_geom,
    'original_segment_count': len(subset)
}
processed_entry

In [ ]:
processed_entry.update(traffic_sums)
processed_groups.append(processed_entry)

In [ ]:
processed_entry

In [ ]:
gdf_processed_entry = gpd.GeoDataFrame(
    [processed_entry],
    geometry=[processed_entry["geometry"]],
    crs="EPSG:3042"
)

gdf_processed_entry_plot = gdf_processed_entry.to_crs(epsg=4326)

In [ ]:
folium.GeoJson(
    gdf_processed_entry_plot.to_json(),
    name="processed_entry",
    style_function=lambda x: {'color': 'green', 'weight': 5, 'opacity': 0.5}
).add_to(m)
m